# Export the v2 model for the browser

Turns `modele_hdfs_v2` (LoRA adapter) into a 4-bit ONNX model that runs in the browser with WebGPU, and uploads it to Hugging Face, where `docs/index.html` loads it from.

A CPU runtime is enough (no GPU needed). It takes about 20-30 minutes, mostly the export.

You need a Hugging Face account and a token with **write** access (huggingface.co > Settings > Access Tokens).

In [ ]:
!pip install -q "transformers==4.46.3" peft accelerate huggingface_hub "optimum-onnx[onnxruntime]" onnx onnxruntime
!git clone -q --depth 1 https://github.com/NafiyTP/RootCause-SLM.git /content/RootCause-SLM
!ls /content/RootCause-SLM/modele_hdfs_v2

## 1. Merge the LoRA adapter into the base model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32)
model = PeftModel.from_pretrained(model, "/content/RootCause-SLM/modele_hdfs_v2").merge_and_unload()
model.save_pretrained("/content/merged", safe_serialization=True)
AutoTokenizer.from_pretrained(BASE).save_pretrained("/content/merged")
del model
!ls -la /content/merged

## 2. Export to ONNX (with KV cache, for fast generation)

In [ ]:
!optimum-cli export onnx --model /content/merged --task text-generation-with-past /content/onnx_fp32
!ls -la /content/onnx_fp32

## 3. Quantize the weights to 4 bits

`model_q4.onnx` (4-bit weights) is what the page loads. The fp32 export (about 6 GB) is not uploaded.

In [ ]:
import os, shutil, glob, json, onnx
from onnxruntime.quantization.matmul_nbits_quantizer import MatMulNBitsQuantizer

WEB = "/content/web"
os.makedirs(f"{WEB}/onnx", exist_ok=True)
m = onnx.load("/content/onnx_fp32/model.onnx")          # also loads model.onnx_data
q = MatMulNBitsQuantizer(m, block_size=32, is_symmetric=True)
q.process()
out = f"{WEB}/onnx/model_q4.onnx"
try:
    onnx.save_model(q.model.model, out)                    # one file if it is under 2 GB
    EXTERNAL = False
except ValueError:
    onnx.save_model(q.model.model, out, save_as_external_data=True,
                    all_tensors_to_one_file=True, location="model_q4.onnx_data")
    EXTERNAL = True
for f in glob.glob("/content/onnx_fp32/*.json") + glob.glob("/content/onnx_fp32/*.txt"):
    shutil.copy(f, WEB)                                     # config, tokenizer, generation config
cfg = json.load(open(f"{WEB}/config.json"))
cfg["transformers.js_config"] = {"dtype": "q4", "use_external_data_format": EXTERNAL}
json.dump(cfg, open(f"{WEB}/config.json", "w"), indent=2)
print("external data:", EXTERNAL)
!ls -la /content/web /content/web/onnx

## 4. Quick check on CPU

The answer should be a JSON with `anomalous`, `cause`, `evidence`, `explanation`.

In [ ]:
import json, sys
sys.path.insert(0, "/content/RootCause-SLM/src")
from blocks_v2 import SYSTEM_PROMPT
from optimum.onnxruntime import ORTModelForCausalLM
tok = AutoTokenizer.from_pretrained(WEB)
ort = ORTModelForCausalLM.from_pretrained(WEB, subfolder="onnx", file_name="model_q4.onnx")
x = json.load(open("/content/RootCause-SLM/data/v2/test_blocks.json"))[0]
msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": x["prompt"]}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
out = ort.generate(ids, max_new_tokens=200, do_sample=False)
print(x["category"], "->", tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True))

## 5. Upload to Hugging Face

The repo is public (the page downloads it from the visitor's browser).

In [ ]:
import getpass
from huggingface_hub import HfApi
api = HfApi(token=getpass.getpass("Hugging Face token (write): "))
user = api.whoami()["name"]
REPO_ID = f"{user}/rootcause-slm-v2-web"
api.create_repo(REPO_ID, exist_ok=True)
api.upload_folder(folder_path=WEB, repo_id=REPO_ID, commit_message="v2 model, ONNX 4-bit for the browser")
print("uploaded to https://huggingface.co/" + REPO_ID)
print("in docs/index.html, SMALL_MODEL must be:", REPO_ID)